# Rotated **N-patch** Joint Lattice-Surgery Measurement — $M\!\left(\prod_i \bar P_i\right)$

One generator, `build_rotated_multi_patch_joint_layout`, builds rotated surface-code multi-patch
joint measurements directly from `PatchSpec` origins.  This notebook demonstrates it on a single
6-patch example,

$$M(\bar Z_1\,\bar Z_2\,\bar X_3\,\bar X_4\,\bar X_5\,\bar X_6)\qquad (\text{4 X-side} + \text{2 Z-side}).$$

The geometry is **derived from coordinates**: the X-patches fuse into one X-side **bus**, and **each
Z-patch attaches through its own mixed (XZ) domain wall**.  The construction measures **only the full
joint** — no single logical, no proper sub-product — leaving $N-1$ logical qubits, and uses genuine
**ancilla-based syndrome extraction** (no MPP).

## What is a legal / illegal patch-placement input?

The input is a list of `PatchSpec(name, origin, distance, measured_logical, orientation)` plus a
`target` saying which logical (`"X"` or `"Z"`) each patch contributes.  The generator first checks
the placement geometrically, then **GF(2)-oracle-gates** it; an illegal input always raises a
concrete `BentLayoutError`/`ValueError` (it never silently mis-measures).

### ✅ Fundamental legality (true of *any* joint-measurement layout)

A placement defines a legal measurement of $\prod_i \bar P_i$ when:

1. **Well-formed coordinates** — origins on the `(odd, odd)` lattice (both `x0, y0` odd), no two
   patches overlap, uniform distance, unique names, and `target` a permutation of the names with each
   Pauli `"X"`/`"Z"`.
2. **The merged/routed region is connected** — one connected data region carries all the patches.
3. **It measures exactly the joint** — the GF(2) oracle confirms $\prod_i \bar P_i$ is in the
   stabilizer span while **no single logical and no proper sub-product** is, leaving `N-1` residual
   logical qubits, with **no weight-1 leftover logical**, all checks commuting, no $Y$/twist, a valid
   DEM and no MPP.
4. **No parallel same-type pair collapses** — two *parallel* same-type measured logicals must be
   separated by an opposite-type wall / junction so their pair-product is not measured (perpendicular
   same-type logicals are fine).  A special case of "no proper sub-product".
5. **Each measured logical is realizable in its declared orientation** — the geometry must host a
   string of that Pauli in that direction; it is never silently re-oriented.

### 🔧 What `routing="auto"` builds

This generator realises the joint on a **bus** — X-side for any joint containing an X-patch, **Z-side
for a pure-Z joint** — with the opposite-type patches attached through **mixed (XZ) walls**:

1. **Pure-X, pure-Z and mixed joints are all supported.**  A pure 2-patch same-type joint
   $M(\bar X_1\bar X_2)$ / $M(\bar Z_1\bar Z_2)$ is a plain same-type merge (no walls; a pure-Z joint
   uses a **Z-memory** circuit).  *Three or more* parallel same-type patches bare-fused over-measure
   their pairwise products and are rejected — a clean $k\ge 3$ same-type parity needs an opposite-type
   **ancilla bus** (future work).
2. The bus-patches connect by **fusion corridors** (sharing a row/column) into a trunk; any unfused
   bus-patch attaches as a **side-arm**.
3. Each opposite-type patch attaches to the bus from **any side** — left, right, above or below —
   through a band that carries its own mixed (XZ) wall.
4. Gaps are even and $\ge 2$ **automatically** (a consequence of odd origins + no overlap — not a
   separate restriction).

### ⛔ Illegal / rejected input (always raises a concrete reason — never silently mis-measured)

1. **An even origin** → `all patch origins must be odd`.
2. **Overlapping patches** → `patches p_i and p_j overlap`.
3. **A same-type collapse** — a parallel same-type pair (or $\ge 3$ same-type patches) whose
   pair-product enters the span → `forbidden placement: the sub-product X3X4 ... is in the stabilizer
   span` (separate them with an opposite-type wall).
4. **An orientation the geometry cannot host** → e.g. `patch p6: its measured X-logical must run
   vertical (orientation='X_vertical'), but no such string commutes with the routed bus here` (flip
   the orientation, or reposition — never silently re-oriented).
5. **A disconnected routed region** → `the routed region is not connected`.
6. **A patch enclosed inside the bus, or not reaching it on any axis** → `... overlap the bus` /
   `does not reach the bus on either axis`.  (A patch *beside / above / below* the bus is fine — only
   one buried inside the trunk is rejected.)

## Coordinate design rules

Data live on the `(odd, odd)` lattice (spacing 2); a distance-$d$ patch at `origin=(x0,y0)` occupies
columns/rows $x_0,\dots,x_0+2(d-1)$ and $y_0,\dots,y_0+2(d-1)$.

* **parity** — all origins odd; **no overlap** between patches.
* **X-side** patches fuse into one region/bus; **each Z-side** patch hangs off the bus by its own
  band, which carries its mixed (XZ) wall.
* **⛔ forbidden pattern (geometric, not a count limit):** two *parallel* same-type measured logicals
  must not close a local sub-product through a **bare** trunk (no opposite-type wall/junction between
  them).  Perpendicular same-type patches are fine — e.g. here $\bar X_5,\bar X_6$ run *vertically*
  on side-arms while $\bar X_3,\bar X_4$ run *horizontally* on the trunk.  If a placement closes a
  sub-product, the generator **raises and names it** (never silently mis-measures).

## Routing API

* `routing="auto"` — the bounded auto-router: derive fusion corridors between fusable X-patches
  (sharing a row/column), grow a vertical bus to reach the Z-patches, attach each Z by its own band,
  and accept the first candidate the GF(2) oracle verifies.
* **explicit routing** (for an upstream lattice-surgery compiler) — pass the routing graph and it is
  honoured **exactly** (no auto-search), still GF(2)-oracle-gated:

  ```python
  routing = {
      "type": "explicit",
      "x_edges": [("p4", "p3"), ("p5", "trunk"), ("p6", "trunk")],   # X–X fusions; (p,"trunk")=side-arm
      "z_attachments": [("p1", "left_wall"), ("p2", "right_wall")],   # asserted vs geometry
  }
  ```

  An illegal explicit routing reports the concrete reason (overlap / orientation / sub-product /
  weight-1 / wrong wall side) — no silent re-orientation, no skipped verification.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import itertools
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon, Circle, Rectangle
from lightstim.qec_code.surface_code.rotated import (
    PatchSpec, build_rotated_multi_patch_joint_layout, BentLayoutError,
    diagnose_multi_patch_joint)
from lightstim.qec_code.surface_code.rotated.bent_layout import _symplectic, _in_span, _gf2_rank


def joint_acceptance(lay):
    '''Re-derive the N-patch joint algebra from scratch (independent of layout.verify()).'''
    sv, n = _symplectic(lay.data)
    S = [sv(ch['pauli']) for ch in lay.checks]
    vecs = [sv({c: P for c in sup}) for _, P, sup in lay.logicals]
    N = len(vecs); joint = np.zeros(2 * n, np.uint8)
    for v in vecs: joint ^= v
    singles = any(_in_span(S, v) for v in vecs)
    subs = []
    for r in range(1, N):
        for comb in itertools.combinations(range(N), r):
            x = np.zeros(2 * n, np.uint8)
            for k in comb: x ^= vecs[k]
            subs.append(_in_span(S, x))
    return dict(joint=_in_span(S, joint), any_single=singles, any_subproduct=any(subs),
                remaining_logical_dof=len(lay.data) - _gf2_rank(S))


def show(lay, title, mark=None):
    '''Generalized N-patch layout drawing: X-side / Z-patch regions, mixed wall, all N logicals.'''
    COL = {'X': '#e23b3b', 'Z': '#2f6fd0', 'M': '#8b3fd0'}
    FILL = {'X': '#f6b8b8', 'Z': '#b8cdf0', 'M': '#d9c2f2'}; GOLD = '#f0a000'
    CHAIN = lay.readout_chain
    allc = [c for ch in lay.checks for c in ch['corners']] + list(lay.data)
    x0, x1 = min(p[0] for p in allc) - 1.6, max(p[0] for p in allc) + 1.6
    y0, y1 = min(p[1] for p in allc) - 1.6, max(p[1] for p in allc) + 1.6
    mcols = sorted({c['syn'][0] for c in lay.checks if c['type'] == 'M'})
    seam = float(np.mean(mcols)) if mcols else (x0 + x1) / 2
    fig, ax = plt.subplots(figsize=(min(13, 0.85 + (x1 - x0) * 0.42), min(13, 0.85 + (y1 - y0) * 0.42)))
    if mcols and (max(mcols) - min(mcols)) <= 4:        # a single wall: shade X-side | Z-patch
        ax.add_patch(Rectangle((x0, y0), seam - x0, y1 - y0, facecolor='#fdeaea', ec='none', zorder=0))
        ax.add_patch(Rectangle((seam, y0), x1 - seam, y1 - y0, facecolor='#e9effb', ec='none', zorder=0))
        ax.text(x0 + 0.3, y0 + 0.5, 'X-side', color='#b02a2a', fontsize=12, fontweight='bold', va='center')
        ax.text(x1 - 0.3, y0 + 0.5, 'Z-patch', color='#234e9e', fontsize=12, fontweight='bold', ha='right', va='center')
    else:                                               # multiple mixed walls (multi-Z): neutral bg
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor='#f6f6f9', ec='none', zorder=0))
    for ch in lay.checks:
        t, pts, syn = ch['type'], ch['corners'], ch['syn']; hl = syn in CHAIN; ec = GOLD if hl else COL[t]
        if len(pts) >= 3:
            cx, cy = np.mean([p[0] for p in pts]), np.mean([p[1] for p in pts])
            order = sorted(pts, key=lambda p: np.arctan2(p[1] - cy, p[0] - cx))
            ax.add_patch(Polygon(order, closed=True, facecolor=FILL[t], edgecolor=ec,
                                 lw=2.4 if hl else 0.6, alpha=0.9 if hl else 0.30, zorder=3 if hl else 2))
        elif len(pts) == 2:
            (a, b), (c, dd) = pts
            ax.plot([a, syn[0], c], [b, syn[1], dd], color=ec, lw=7 if hl else 5,
                    alpha=0.7 if hl else 0.28, solid_capstyle='round', zorder=3 if hl else 2)
        if t == 'M':
            for c, P in ch['pauli'].items():
                ax.plot([syn[0], c[0]], [syn[1], c[1]], color=COL[P], lw=2.4, zorder=5, alpha=0.9, solid_capstyle='round')
        ax.add_patch(Rectangle((syn[0] - 0.18, syn[1] - 0.18), 0.36, 0.36, facecolor=COL[t],
                     edgecolor=GOLD if hl else 'white', lw=2 if hl else 0.8, zorder=6, alpha=1 if hl else 0.55))
    for q in lay.data:
        ax.add_patch(Circle(q, 0.14, facecolor='#1a1a1a', edgecolor='white', lw=0.6, zorder=8))
    xshades = ['#c01616', '#7a0d0d', '#e0552a', '#9c1b5a', '#5a1b9c']; xi = 0
    for nm, P, sup in lay.logicals:
        col = '#13346e' if P == 'Z' else xshades[xi % len(xshades)]; xi += (P == 'X')
        s = sorted(sup)
        ax.plot([q[0] for q in s], [q[1] for q in s], color=col, lw=5, zorder=10, solid_capstyle='round')
        mx, my = s[len(s) // 2]
        ax.text(mx, my - 0.8, fr'$\bar {P}_{{{nm[1:]}}}$', color=col, fontsize=13, fontweight='bold', ha='center', zorder=11,
                bbox=dict(boxstyle='round,pad=0.12', fc='white', ec=col, lw=1))
    handles = [mpatches.Patch(color=COL['X'], label='X stabilizer'), mpatches.Patch(color=COL['Z'], label='Z stabilizer'),
               mpatches.Patch(color=COL['M'], label='MIXED (XZ) wall'),
               mpatches.Patch(facecolor='#fff3d6', edgecolor=GOLD, lw=2, label='readout chain (product = joint)')]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.005), ncol=2, fontsize=9)
    if mark:                                              # weight-1 leftover logicals (green)
        for q, P, cls in mark:
            ax.add_patch(Circle(q, 0.55, fill=False, ec='#16a34a', lw=2.6, zorder=14))
            ax.text(q[0], q[1] + 1.0, f'{P}@{q}→{cls}', color='#0a7a30', fontsize=8,
                    fontweight='bold', ha='center', zorder=15,
                    bbox=dict(boxstyle='round,pad=0.1', fc='#eafbea', ec='#16a34a', lw=1))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    ax.set_title(title, fontsize=12, pad=46); plt.tight_layout(); plt.show()


def run_multi_patch_joint_case(patches, target=None, title="N-patch joint", draw=True,
                               rounds=None, routing="auto"):
    '''Build + summarize + verify + circuit-check + visualize an N-patch joint in ONE call.

    Returns (layout, result).  ``result`` merges the eleven verify() checks, the independently
    re-derived joint algebra, and an overall ``all_pass``.  An un-routable placement is reported
    (with the BentLayoutError reason) and returns (None, {... 'all_pass': False}).
    '''
    print(f"=== {title} ===")
    try:
        lay = build_rotated_multi_patch_joint_layout(patches, target=target, routing=routing)
    except (BentLayoutError, ValueError, NotImplementedError) as e:
        print(f"  NOT ROUTABLE — {type(e).__name__}: {e}")
        return None, {"built": False, "error": str(e), "all_pass": False}
    nX = sum(c['type'] == 'X' for c in lay.checks); nZ = sum(c['type'] == 'Z' for c in lay.checks)
    nM = sum(c['type'] == 'M' for c in lay.checks)
    print(f"  N={lay.N} patches   data={len(lay.data)}   stabilizers={len(lay.checks)} "
          f"(X={nX}, Z={nZ}, MIXED={nM})")
    for nm, P, sup in lay.logicals:
        print(f"    {nm}: {P}̄  on {sorted(sup)}")
    acc = joint_acceptance(lay)
    print(f"  joint ∏P̄ measured = {acc['joint']}   no single = {not acc['any_single']}   "
          f"no proper sub-product = {not acc['any_subproduct']}   "
          f"remaining logical d.o.f. = {acc['remaining_logical_dof']}  (expect N-1 = {lay.N - 1})")
    res = lay.verify(rounds=rounds)
    print("  verify (11): " + "  ".join(f"{k}={'T' if v else 'F'}" for k, v in res.items()))
    circ = lay.build_circuit(rounds=rounds or lay.distance, p=0.0)
    cxcz = any(i.name == 'CX' for i in circ.flattened()) and any(i.name == 'CZ' for i in circ.flattened())
    print(f"  circuit: qubits={circ.num_qubits}  detectors={circ.num_detectors}  "
          f"no_MPP={'MPP' not in str(circ)}  DEM_valid={res['dem_valid']}  "
          f"deterministic={res['no_mpp'] and res['dem_valid']}  CX+CZ={cxcz}")
    result = {**res,
              "joint_ok": acc['joint'] and not acc['any_single'] and not acc['any_subproduct'],
              "remaining_dof_ok": acc['remaining_logical_dof'] == lay.N - 1, "built": True}
    result["all_pass"] = all(res.values()) and result["joint_ok"] and result["remaining_dof_ok"]
    print(f"  ALL PASS = {result['all_pass']}")
    if draw:
        show(lay, f"{title}  —  {len(lay.data)} data, {len(lay.checks)} stab, {lay.N - 1} logical")
    return lay, result

print("ready — call run_multi_patch_joint_case(patches, title=...)")


def show_candidate(diag, title, note=None, mark=None):
    """Draw a candidate routing from diagnose_multi_patch_joint() — works even when it FAILS.

    Tiles are the candidate stabilizers (red=X, blue=Z, purple=mixed wall); the gold plaquettes are
    the chain whose GF(2) product equals the highlighted operator (the full joint when the layout is
    clean, otherwise the offending sub-product).  The sub-product's patches are boxed."""
    COL = {'X': '#e23b3b', 'Z': '#2f6fd0', 'M': '#8b3fd0'}
    FILL = {'X': '#f6b8b8', 'Z': '#b8cdf0', 'M': '#d9c2f2'}; GOLD = '#f0a000'
    if diag.ok:
        chain, emph = diag.joint_chain, set()
        chlabel = 'chain: product = full joint'; verdict = 'PASS — full joint only, no sub-product'; vc = '#177a17'
    elif diag.failing_subproducts:
        lbl, names, chain = diag.failing_subproducts[0]; emph = set(names)
        chlabel = f'chain: product = {lbl}  (the sub-product!)'; verdict = f'FAIL — {lbl} is in span'; vc = '#c00'
    else:
        chain, emph = set(), set(); chlabel = '(no chain)'; verdict = 'FAIL — joint not hostable'; vc = '#c00'
    syn2p = diag.syn_to_plaq
    allc = [c for p in diag.plaqs for c in p['corners']] + list(diag.data)
    x0, x1 = min(p[0] for p in allc) - 1.6, max(p[0] for p in allc) + 1.6
    y0, y1 = min(p[1] for p in allc) - 1.6, max(p[1] for p in allc) + 1.6
    fig, ax = plt.subplots(figsize=(min(12, 0.9 + (x1 - x0) * 0.4), min(12, 1.0 + (y1 - y0) * 0.4)))
    for p in diag.plaqs:                                   # candidate stabilizer tiles
        t, pts = p['type'], p['corners']
        if len(pts) >= 3:
            cx, cy = np.mean([q[0] for q in pts]), np.mean([q[1] for q in pts])
            order = sorted(pts, key=lambda q: np.arctan2(q[1] - cy, q[0] - cx))
            ax.add_patch(Polygon(order, closed=True, facecolor=FILL[t], edgecolor=COL[t],
                                 lw=0.5, alpha=0.85 if t == 'M' else 0.30))
    for s in chain:                                        # the closing chain (gold)
        p = syn2p.get(s)
        if not p: continue
        pts = p['corners']
        if len(pts) >= 3:
            cx, cy = np.mean([q[0] for q in pts]), np.mean([q[1] for q in pts])
            order = sorted(pts, key=lambda q: np.arctan2(q[1] - cy, q[0] - cx))
            ax.add_patch(Polygon(order, closed=True, facecolor='#fff0cf', edgecolor=GOLD, lw=2.6, alpha=0.93, zorder=4))
        elif len(pts) == 2:
            (a, b), (c, dd) = pts
            ax.plot([a, s[0], c], [b, s[1], dd], color=GOLD, lw=6, alpha=0.85, zorder=4, solid_capstyle='round')
    for q in diag.data:
        ax.add_patch(Circle(q, 0.14, facecolor='#1a1a1a', ec='white', lw=0.5, zorder=8))
    for nm, P, sup in diag.logicals:                      # the four logicals
        s = sorted(sup); is_e = nm in emph; col = {'X': '#b00', 'Z': '#039'}[P]
        ax.plot([q[0] for q in s], [q[1] for q in s], color=col, lw=6.5 if is_e else 4, zorder=10, solid_capstyle='round')
        mx, my = s[len(s) // 2]
        ax.text(mx, my - 0.85, fr'$\bar {P}_{{{nm[1:]}}}$', color=col, fontsize=14, fontweight='bold',
                ha='center', zorder=11, bbox=dict(boxstyle='round,pad=0.12', fc='#fff2b0' if is_e else 'white',
                ec=col, lw=2 if is_e else 1))
    if emph:                                              # box the collapsing same-type patches
        ec = [(x, y) for nm, P, sup in diag.logicals if nm in emph for (x, y) in sup]
        bx0, bx1 = min(x for x, _ in ec) - 0.7, max(x for x, _ in ec) + 0.7
        by0, by1 = min(y for _, y in ec) - 0.7, max(y for _, y in ec) + 0.7
        ax.add_patch(Rectangle((bx0, by0), bx1 - bx0, by1 - by0, fill=False, ec='#c00', lw=2, ls='--', zorder=12))
    handles = [mpatches.Patch(color=COL['X'], label='X stab'), mpatches.Patch(color=COL['Z'], label='Z stab'),
               mpatches.Patch(color=COL['M'], label='mixed (XZ) wall'),
               mpatches.Patch(facecolor='#fff0cf', edgecolor=GOLD, label=chlabel)]
    ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.17), ncol=4, fontsize=7.5)
    if note:
        ax.text(0.5, -0.02, note, transform=ax.transAxes, ha='center', va='top', fontsize=9.5, color='#333', wrap=True)
    if mark:                                              # weight-1 leftover logicals (green)
        for q, P, cls in mark:
            ax.add_patch(Circle(q, 0.55, fill=False, ec='#16a34a', lw=2.6, zorder=14))
            ax.text(q[0], q[1] + 1.0, f'{P}@{q}→{cls}', color='#0a7a30', fontsize=8,
                    fontweight='bold', ha='center', zorder=15,
                    bbox=dict(boxstyle='round,pad=0.1', fc='#eafbea', ec='#16a34a', lw=1))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect('equal'); ax.invert_yaxis(); ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold', color=vc, pad=40); plt.tight_layout(); plt.show()

print("diagnostic drawer ready — show_candidate(diagnose_multi_patch_joint(...))")


def weight1_logicals(lay):
    """Single-qubit ops that commute with every selected stabilizer but are not stabilizers — i.e.
    weight-1 leftover logicals (code distance 1 at that qubit).  Returns [(qubit, Pauli)]."""
    sv, n = _symplectic(lay.data); S = [sv(c['pauli']) for c in lay.checks]
    out = []
    for q in lay.data:
        for P in 'XZ':
            v = sv({q: P})
            comm = all(((v[:n] & s[n:]).sum() + (v[n:] & s[:n]).sum()) % 2 == 0 for s in S)
            if comm and not _in_span(S, v):
                out.append((q, P))
    return out


def homology_of(lay, q, P):
    """Which product of the patch logicals the single-qubit op P@q is homologous to (label)."""
    sv, n = _symplectic(lay.data); S = [sv(c['pauli']) for c in lay.checks]
    v = sv({q: P}); Lv = [sv({c: Q for c in sup}) for _, Q, sup in lay.logicals]
    nm = [f'{Q}{name[1:]}' for name, Q, sup in lay.logicals]
    for r in range(len(Lv) + 1):
        for comb in itertools.combinations(range(len(Lv)), r):
            x = v.copy()
            for k in comb:
                x ^= Lv[k]
            if _in_span(S, x):
                return ''.join(nm[k] for k in comb) or '(stabilizer)'
    return '?'

## The 6-patch example — $M(\bar Z_1\bar Z_2\bar X_3\bar X_4\bar X_5\bar X_6)$

A vertical X-trunk carries $\bar X_4$ (top) and $\bar X_3$ (bottom); $\bar X_5,\bar X_6$ are
perpendicular side-arms on the left; $\bar Z_1$ (upper-left) and $\bar Z_2$ (right) attach through
their own mixed walls.

In [ ]:
patches_4x2z = [
    PatchSpec("p4", (11, 1),  3, "X", "X_horizontal"),   # X̄4  top of trunk
    PatchSpec("p3", (11, 29), 3, "X", "X_horizontal"),   # X̄3  bottom of trunk
    PatchSpec("p5", (3, 23),  3, "X", "X_vertical"),     # X̄5  lower-left side-arm
    PatchSpec("p6", (3, 17),  3, "X", "X_vertical"),     # X̄6  left side-arm
    PatchSpec("p1", (3, 9),   3, "Z", "X_horizontal"),   # Z̄1
    PatchSpec("p2", (19, 17), 3, "Z", "X_horizontal"),   # Z̄2
]
target_4x2z = [("p1","Z"), ("p2","Z"), ("p3","X"), ("p4","X"), ("p5","X"), ("p6","X")]

lay = build_rotated_multi_patch_joint_layout(patches_4x2z, target=target_4x2z, routing="auto")

# ---- verification table ----
acc = joint_acceptance(lay)
res = lay.verify()
sv, n = _symplectic(lay.data); S = [sv(c["pauli"]) for c in lay.checks]
vecs = [sv({c: P for c in sup}) for _, P, sup in lay.logicals]; N = len(vecs)
nsub = sum(_in_span(S, np.bitwise_xor.reduce([vecs[k] for k in comb]))
           for r in range(1, N) for comb in itertools.combinations(range(N), r))
w1 = weight1_logicals(lay)
print(f"M(Z1 Z2 X3 X4 X5 X6)   N={lay.N} patches   data={len(lay.data)}   stabilizers={len(lay.checks)}")
print(f"  full joint in span          : {acc['joint']}")
print(f"  proper sub-products in span  : {nsub}")
print(f"  remaining logical dof        : {acc['remaining_logical_dof']}  (want N-1 = {lay.N-1})")
print(f"  no weight-1 leftover logical : {res['no_weight1_logical']}" + ("" if not w1 else f"  {w1}"))
print(f"  all stabilizers commute      : {res['commute']}    no Y / no twist : {res['no_twist']}")
print(f"  no MPP                       : {res['no_mpp']}")
print(f"  DEM valid                    : {res['dem_valid']}")
print(f"  no tick collision            : {res['no_tick_collision']}")
print(f"  ALL 11 verify checks pass    : {all(res.values())}")

### Layout — data qubits, X / Z stabilizers, mixed (XZ) walls, readout chain
The gold chain is the set of check syndromes whose GF(2) product equals the full 6-body joint.

In [ ]:
show(lay, "M(Z1 Z2 X3 X4 X5 X6) — layout, stabilizers, mixed walls, readout chain")

### Stim detector-slice diagram (ancilla-based syndrome extraction, no MPP)
`circuit.diagram("detslice-with-ops-svg")` shows the actual gate-level circuit: data↔ancilla `CX`/`CZ`
and ancilla measurements `MR` — i.e. **real ancilla-based syndrome extraction, not `MPP`** — with the
detector slices overlaid.  (Shown for 2 rounds; `verify()` uses the full $d$ rounds.)

In [ ]:
circuit = lay.build_circuit(rounds=2, p=0.0)
print("no MPP        :", "MPP" not in str(circuit))
print("uses CX + CZ  :", any(i.name=='CX' for i in circuit.flattened()) and any(i.name=='CZ' for i in circuit.flattened()))
print("qubits        :", circuit.num_qubits, " detectors:", circuit.num_detectors)
print("DEM valid     :", circuit.detector_error_model(decompose_errors=True).num_detectors == circuit.num_detectors)
from IPython.display import SVG
SVG(str(circuit.diagram("detslice-with-ops-svg")))  # SVG-wrap avoids a duplicate text/plain copy

### Explicit routing (what an upstream compiler would pass)
The same layout via an **explicit** routing graph — taken exactly as given (no auto-search), still
oracle-verified.

In [ ]:
lay_explicit = build_rotated_multi_patch_joint_layout(
    patches_4x2z, target=target_4x2z,
    routing={
        "type": "explicit",
        "x_edges": [("p4", "p3"), ("p5", "trunk"), ("p6", "trunk")],
        "z_attachments": [("p1", "left_wall"), ("p2", "right_wall")],
    })
print("explicit routing — ALL 11 verify checks pass:", all(lay_explicit.verify().values()))
print("measures only the full joint (no sub-product):", joint_acceptance(lay_explicit)['any_subproduct'] is False)

## The same 6-patch joint at $d=5$

`make_6patch(d)` scales the example to any distance ($d=3$ above is `make_6patch(3)`).  At $d=5$ the
routed region has 255 data qubits and the build (selection + swap-repair) takes $\approx 90$ s — the
*same* construction, a larger code.

In [ ]:
def make_6patch(d):
    '''The 6-patch M(Z1 Z2 X3 X4 X5 X6) parametrized by distance (d=3 == the example above).'''
    s = 2 * (d - 1)
    LX = 3; TX = LX + s + 4; RX = TX + s + 4          # left col / trunk col / right col
    r1 = 1 + s + 4; r6 = r1 + s + 4; r5 = r6 + s + 2; r3 = r5 + s + 2   # stacked rows (p2 opposite p6)
    patches = [
        PatchSpec("p4", (TX, 1),  d, "X", "X_horizontal"), PatchSpec("p3", (TX, r3), d, "X", "X_horizontal"),
        PatchSpec("p5", (LX, r5), d, "X", "X_vertical"),   PatchSpec("p6", (LX, r6), d, "X", "X_vertical"),
        PatchSpec("p1", (LX, r1), d, "Z", "X_horizontal"), PatchSpec("p2", (RX, r6), d, "Z", "X_horizontal"),
    ]
    target = [("p1", "Z"), ("p2", "Z"), ("p3", "X"), ("p4", "X"), ("p5", "X"), ("p6", "X")]
    return patches, target

lay5 = build_rotated_multi_patch_joint_layout(*make_6patch(5))   # ~90 s (built once; reused below)
res5 = lay5.verify()
print(f"d=5:  N={lay5.N}  data={len(lay5.data)}  stabilizers={len(lay5.checks)}")
print(f"  no proper sub-product : {res5['no_subjoint']}     residual dof = N-1 : {res5['logical_count']}")
print(f"  no weight-1 leftover  : {res5['no_weight1_logical']}     no MPP / DEM valid / no tick : "
      f"{res5['no_mpp']} / {res5['dem_valid']} / {res5['no_tick_collision']}")
print(f"  ALL 11 verify checks pass : {all(res5.values())}")

### $d=5$ layout — data qubits, X / Z stabilizers, mixed walls, readout chain

In [ ]:
show(lay5, "M(Z1 Z2 X3 X4 X5 X6) at d=5 — layout, stabilizers, mixed walls, readout chain")

### $d=5$ Stim detector-slice diagram (ancilla-based, no MPP)
Same `detslice-with-ops-svg` view as $d=3$, now for the $d=5$ code (a large SVG — 2 rounds).

In [ ]:
circuit5 = lay5.build_circuit(rounds=2, p=0.0)
print("no MPP:", "MPP" not in str(circuit5),
      " | uses CX+CZ:", any(i.name=='CX' for i in circuit5.flattened()) and any(i.name=='CZ' for i in circuit5.flattened()),
      " | qubits:", circuit5.num_qubits, " detectors:", circuit5.num_detectors,
      " | DEM valid:", circuit5.detector_error_model(decompose_errors=True).num_detectors == circuit5.num_detectors)
from IPython.display import SVG
SVG(str(circuit5.diagram("detslice-with-ops-svg")))

## Logical error rate vs physical error rate

A decoding sanity check for the 6-patch joint $M(\bar Z_1\bar Z_2\bar X_3\bar X_4\bar X_5\bar X_6)$:
circuit-level depolarizing noise at strength $p$, `rounds = d`, PyMatching decoder.  Below threshold
the logical error rate is strongly suppressed as $p$ decreases — confirming the routed merge is
genuinely error-correcting.

Shown for **$d=3$ and $d=5$** on one log-log plot (the $d=5$ layout takes $\approx 90$ s to build).
The two curves **cross near $p\approx 5\times10^{-3}$** (this construction's circuit-level
threshold): **below** it the $d=5$ curve is at/under $d=3$ (distance suppression — the goal); **above**
it the larger code is worse, as expected.  (The $d=5$ points at $p=1,2\times10^{-3}$ saw 0 errors in the
demo shot budget, so they are off the log axis.)  Even higher
distances ($d=7+$) and high-statistics sweeps are slow and belong in a **benchmark script**, not this
demo — `make_6patch(d)` is general, run them offline.

In [ ]:
import pymatching

def ler(lay, d, p, max_shots=int(1e6), max_errors=100, batch=50000):
    '''Circuit-level LER of the joint-measurement memory experiment, decoded with PyMatching.
    (Demo budget; for high-statistics / larger d use max_shots=1e8, max_errors=500 in a benchmark script.)'''
    circ = lay.build_circuit(rounds=d, p=p)
    matcher = pymatching.Matching.from_detector_error_model(circ.detector_error_model(decompose_errors=True))
    sampler = circ.compile_detector_sampler()
    shots = errors = 0
    while shots < max_shots and errors < max_errors:
        nrun = min(batch, max_shots - shots)
        dets, obs = sampler.sample(nrun, separate_observables=True)
        errors += int(np.sum(np.any(matcher.decode_batch(dets) != obs, axis=1)))
        shots += nrun
    return errors / shots

distances = [3, 5]
ps = [2e-3, 3e-3, 5e-3, 7e-3, 1e-2]
plt.figure(figsize=(5, 5))
for d, lay_d, mk in [(3, lay, "o"), (5, lay5, "s")]:        # reuse the layouts built above (no rebuild)
    ys = [ler(lay_d, d, p) for p in ps]                     # max_shots=1e6, max_errors=100
    xp = [p for p, y in zip(ps, ys) if y > 0]; yp = [y for y in ys if y > 0]   # mask 0 for the log axis
    plt.loglog(xp, yp, marker=mk, label=f"d = {d}")
    print(f"d={d}: " + "   ".join(f"{p:.0e}->{y:.2e}" for p, y in zip(ps, ys)))

plt.xlabel("Physical error rate $p$")
plt.ylabel("Logical error rate (LER)")
plt.title(r"6-patch joint $M(\bar Z_1\bar Z_2\bar X_3\bar X_4\bar X_5\bar X_6)$ — LER vs $p$")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(title="Distance")
plt.tight_layout()
plt.show()